In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# # for dirname, _, filenames in os.walk('/kaggle/input'):
# #     for filename in filenames:
# #         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

BASE_PATH = '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/'
TRAIN_CSV = os.path.join(BASE_PATH, 'train.csv')
TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
IMAGE_DIR = os.path.join(BASE_PATH, 'images/')

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Define the 20 classes [cite: 16]
class_names = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 
    'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'Pneumoperitoneum', 
    'Pneumomediastinum', 'Subcutaneous Emphysema', 'Tortuous Aorta', 
    'Calcification of the Aorta', 'No Finding'
]

# 2. Load the training CSV [cite: 94, 96]
train_df = pd.read_csv('/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/train.csv')

# 3. Create a helper column for stratification
# This finds the index of the "1" for each row [cite: 83, 85]
train_df['label_idx'] = np.argmax(train_df[class_names].values, axis=1)

# 4. Perform the split (80% train, 20% validation)
train_data, val_data = train_test_split(
    train_df, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_df['label_idx']
)

print(f"train_data defined: {len(train_data)} samples")
print(f"val_data defined: {len(val_data)} samples")

train_data defined: 40834 samples
val_data defined: 10209 samples


In [4]:
# --- 3. Create the Dataset Class ---
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform
        self.labels = self.df[class_names].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Image ID is in the 'id' column
        img_name = self.df.iloc[idx]['id']
        img_path = os.path.join(self.img_dir, img_name)
        
        # Load image; convert to RGB for transfer learning compatibility
        image = Image.open(img_path).convert('RGB')
        
        # Target: get the index of the class that is '1' [cite: 85]
        label = np.argmax(self.labels[idx])
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# to be changed


In [5]:
# # --- 4. Basic Transforms for the GPU ---
# # We resize to 224x224 (standard for models like ResNet/DenseNet)
# data_transforms = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# ])

# # Initialize Dataset
# train_dataset = ChestXrayDataset(train_df, IMAGE_DIR, transform=data_transforms)

# print(f"\nSuccessfully initialized dataset with {len(train_dataset)} images.")

In [6]:
from torchvision import transforms
import torch.nn as nn

# --- Updated Transforms ---
# Training: Add variability to help with rare classes
train_transforms = transforms.Compose([
    transforms.Resize((320, 320)), #earlier it was 224*224
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.RandomRotation(15), # X-rays can be slightly tilted
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Handle different X-ray exposures
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation/Test: No augmentation, just standardization
val_transforms = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- Re-assign to DataLoaders ---
train_dataset = ChestXrayDataset(train_data, IMAGE_DIR, transform=train_transforms)
val_dataset = ChestXrayDataset(val_data, IMAGE_DIR, transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True
)

print("Data Augmentation applied to Training Set.")

Data Augmentation applied to Training Set.


In [7]:
# # --- 2. Define Weighted Loss Function ---
# # Calculate weights inversely proportional to frequency
# # Formula: weight = total_samples / (num_classes * class_count)
# class_counts_array = train_df[class_names].sum().values
# total_samples = len(train_df)
# weights = total_samples / (len(class_names) * class_counts_array)

# # To strictly follow the competition's -5 penalty for False Negatives, 
# # we further boost the weight of all 'Disease' classes relative to 'No Finding'.
# # Index 19 is 'No Finding' based on our class_names list.
# weights[:-1] = weights[:-1] * 5.0 

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# class_weights = torch.FloatTensor(weights).to(device)

# # Using CrossEntropyLoss with weights
# criterion = nn.CrossEntropyLoss(weight=class_weights)

# # --- 3. Create DataLoaders with GPU optimizations ---
# train_loader = DataLoader(
#     ChestXrayDataset(train_data, IMAGE_DIR, transform=data_transforms),
#     batch_size=32, shuffle=True, num_workers=4, pin_memory=True
# )

# val_loader = DataLoader(
#     ChestXrayDataset(val_data, IMAGE_DIR, transform=data_transforms),
#     batch_size=32, shuffle=False, num_workers=4, pin_memory=True
# )

# print("DataLoaders ready for GPU acceleration.")

In [8]:
import torchvision.models as models
from torch.cuda.amp import GradScaler, autocast

# --- 1. Initialize Model (DenseNet121) ---
model = models.densenet121(pretrained=True)

# Replace the last linear layer (classifier)
# DenseNet121 has 1024 input features in its last layer
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 20) 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# --- 2. Optimizer and Scaler ---
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scaler = GradScaler() # For Mixed Precision GPU acceleration

# --- 3. Training Function ---
def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # GPU Acceleration: Mixed Precision
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
    
    return running_loss / len(loader.dataset)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 30.8M/30.8M [00:00<00:00, 195MB/s]
/tmp/ipykernel_23/3963578760.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # For Mixed Precision GPU acceleration


In [9]:
# --- 4. Validation Function ---
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            # Get the class with highest probability
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return running_loss / len(loader.dataset), all_preds, all_labels

print("Model and Training functions are ready.")

Model and Training functions are ready.


In [10]:
# epochs = 5
# for epoch in range(epochs):
#     train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
#     val_loss, preds, labels = validate(model, val_loader, criterion, device)
    
#     print(f"Epoch {epoch+1}/{epochs}.. "
#           f"Train loss: {train_loss:.4f}.. "
#           f"Val loss: {val_loss:.4f}")

In [11]:
# import numpy as np

# def calculate_competition_score(all_preds, all_labels, class_names):
#     all_preds = np.array(all_preds)
#     all_labels = np.array(all_labels)
#     class_scores = []
    
#     print(f"{'Pathology':<25} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Score_c'}")
#     print("-" * 65)
    
#     for i, class_name in enumerate(class_names):
#         tp = np.sum((all_preds == i) & (all_labels == i)) 
#         fp = np.sum((all_preds == i) & (all_labels != i)) 
#         fn = np.sum((all_preds != i) & (all_labels == i))
#         nc = np.sum(all_labels == i)
#         if nc > 0:
#             # Formula: (TP - FP - 5*FN) / N
#             score_c = (tp - fp - 5 * fn) / nc
#             class_scores.append(score_c)
#             print(f"{class_name:<25} | {tp:<5} | {fp:<5} | {fn:<5} | {score_c:.4f}")
#         else:
#             print(f"{class_name:<25} | No samples")

#     # Final Score is the average of all class scores
#     final_score = np.mean(class_scores)
#     print("-" * 65)
#     print(f"FINAL COMPETITION SCORE (Macro): {final_score:.5f}")
#     return final_score

# # Run validation and score
# val_loss, val_preds, val_labels = validate(model, val_loader, criterion, device)
# final_val_score = calculate_competition_score(val_preds, val_labels, class_names)

# RUN 2

In [12]:
import torch.nn.functional as F

# --- 2. Define Weighted Loss Function ---
# Calculate weights inversely proportional to frequency
# Formula: weight = total_samples / (num_classes * class_count)
class_counts_array = train_df[class_names].sum().values
total_samples = len(train_df)
# weights = total_samples / (len(class_names) * class_counts_array //for 1st subsmission

# Use square root to dampen the effect of extreme imbalance
weights = np.sqrt(total_samples / (class_counts_array + 1e-6)) #for submission 2

# Normalize weights so the mean is 1.0
weights = weights / np.mean(weights)
# To strictly follow the competition's -5 penalty for False Negatives, 
# we further boost the weight of all 'Disease' classes relative to 'No Finding'.
# Index 19 is 'No Finding' based on our class_names list
#weights[:-1] = weights[:-1] * 4.0 #For 1st submission
#......using a sqrt or log weight func.......
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.FloatTensor(weights).to(device)

# Using CrossEntropyLoss with weights
criterion = nn.CrossEntropyLoss(weight=class_weights)
# #-------------------------------------------------------------------------------------------------------------
# #for submission 3

# # class FocalLoss(nn.Module):
# #     def __init__(self, alpha=1, gamma=2.0, weight=None):
# #         super(FocalLoss, self).__init__()
# #         self.gamma = gamma
# #         self.weight = weight # Keep your class weights here too

# #     def forward(self, inputs, targets):
# #         ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
# #         pt = torch.exp(-ce_loss)
# #         focal_loss = (alpha * (1 - pt) ** self.gamma * ce_loss).mean()
# #         return focal_loss

# # # Use this in your training setup
# # criterion = FocalLoss(alpha=1, gamma=2.0, weight=class_weights)

# --- 3. Create DataLoaders with GPU optimizations ---
train_loader = DataLoader(
    ChestXrayDataset(train_data, IMAGE_DIR, transform=train_transforms),
    batch_size=32, shuffle=True, num_workers=4, pin_memory=True
)

val_loader = DataLoader(
    ChestXrayDataset(val_data, IMAGE_DIR, transform=val_transforms),
    batch_size=32, shuffle=False, num_workers=4, pin_memory=True
)

print("DataLoaders ready for GPU acceleration.")


# # import torch.nn.functional as F

# # class FocalLoss(nn.Module):
# #     def __init__(self, weight=None, gamma=2.0, alpha=1.0):
# #         super(FocalLoss, self).__init__()
# #         self.gamma = gamma
# #         self.weight = weight 
# #         self.alpha = alpha # Correctly store alpha

# #     def forward(self, inputs, targets):
# #         # ce_loss is calculated for each sample in the batch
# #         ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
# #         pt = torch.exp(-ce_loss)
# #         # Apply the focal scaling factor
# #         focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()
# #         return focal_loss

# # # 1. Calculate weights for the Loss Function
# # class_counts_array = train_data[class_names].sum().values # Use train_data for consistency
# # total_samples = len(train_data)

# # # Square root scaling prevents rare classes from over-dominating and causing FPs
# # weights = np.sqrt(total_samples / (class_counts_array + 1e-6))
# # weights = weights / np.mean(weights)

# # # Assign to device
# # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# # class_weights_tensor = torch.FloatTensor(weights).to(device)

# # # Initialize Focal Loss
# # criterion = FocalLoss(weight=class_weights_tensor, gamma=2.0, alpha=1.0)

# # from torch.utils.data import WeightedRandomSampler

# # # 1. Calculate the weight for each class (1/frequency)
# # # 'label_idx' contains the index of the 1 for each row [cite: 83, 85]
# # counts = train_data['label_idx'].value_counts().sort_index().values
# # class_weights_sampling = 1.0 / (counts + 1e-6)

# # # 2. Map those weights to every sample in the training dataframe
# # sample_weights = [class_weights_sampling[t] for t in train_data['label_idx']]

# # # 3. Create the sampler
# # sampler = WeightedRandomSampler(
# #     weights=sample_weights, 
# #     num_samples=len(sample_weights), 
# #     replacement=True
# # )

# # # 4. Create DataLoaders
# # # IMPORTANT: Remove 'shuffle=True' when using a sampler
# # train_loader = DataLoader(
# #     ChestXrayDataset(train_data, IMAGE_DIR, transform=train_transforms),
# #     batch_size=32, 
# #     sampler=sampler, # This forces the model to see rare classes every batch
# #     num_workers=4, 
# #     pin_memory=True
# # )

# # val_loader = DataLoader(
# #     ChestXrayDataset(val_data, IMAGE_DIR, transform=val_transforms),
# #     batch_size=32, 
# #     shuffle=False, 
# #     num_workers=4, 
# #     pin_memory=True
# # )

# # print("Sureshot Sampling and Focal Loss implemented.")

DataLoaders ready for GPU acceleration.


In [13]:
# import torch.nn.functional as F
# import torch.nn as nn

# # 1. Calculate Weights based on training data distribution [cite: 19, 20]
# class_counts_array = train_data[class_names].sum().values 
# total_samples = len(train_data)

# # Use square root to dampen the effect of extreme imbalance [cite: 25, 26]
# weights = np.sqrt(total_samples / (class_counts_array + 1e-6)) 

# # Normalize weights so the mean is 1.0 to keep gradients stable
# weights = weights / np.mean(weights)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# class_weights = torch.FloatTensor(weights).to(device)

# # 2. Corrected Focal Loss Definition [cite: 28, 129]
# class FocalLoss(nn.Module):
#     def __init__(self, alpha=1.0, gamma=2.0, weight=None):
#         super(FocalLoss, self).__init__()
#         self.alpha = alpha
#         self.gamma = gamma
#         self.weight = weight 

#     def forward(self, inputs, targets):
#         # Calculate standard cross entropy first
#         ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
        
#         # Calculate probability of the correct class (pt)
#         pt = torch.exp(-ce_loss)
        
#         # Apply the Focal Loss formula: (alpha * (1-pt)^gamma * CE)
#         # This reduces loss for easy samples and increases it for hard ones [cite: 23, 24]
#         focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()
        
#         return focal_loss

# # 3. Initialize the Focal Loss criterion
# # Gamma=2.0 is the standard starting point for medical class imbalance [cite: 124, 129]
# criterion = FocalLoss(alpha=1.0, gamma=2.0, weight=class_weights)

# # --- 3. Create DataLoaders with GPU optimizations ---
# # For sureshot success, ensure train_transforms uses at least 320x320 resolution
# train_loader = DataLoader(
#     ChestXrayDataset(train_data, IMAGE_DIR, transform=train_transforms),
#     batch_size=32, shuffle=True, num_workers=4, pin_memory=True
# )

# val_loader = DataLoader(
#     ChestXrayDataset(val_data, IMAGE_DIR, transform=val_transforms),
#     batch_size=32, shuffle=False, num_workers=4, pin_memory=True
# )

# print("Criterion updated to Focal Loss. DataLoaders ready.")

In [14]:
# --- MOVE THIS BEFORE THE TRAINING LOOP ---
def check_improvement(all_preds, all_labels, class_names):
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    class_scores = []
    
    # Header for the breakdown
    print(f"{'Pathology':<25} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Score_c'}")
    print("-" * 65)
    
    for i, class_name in enumerate(class_names):
        tp = np.sum((all_preds == i) & (all_labels == i))
        fp = np.sum((all_preds == i) & (all_labels != i))
        fn = np.sum((all_preds != i) & (all_labels == i)) 
        nc = np.sum(all_labels == i) 
        
        if nc > 0:
            # Official Formula: (TP - FP - 5*FN) / N [cite: 61]
            score_c = (tp - fp - 5 * fn) / nc
            class_scores.append(score_c)
            print(f"{class_name:<25} | {tp:<5} | {fp:<5} | {fn:<5} | {score_c:.4f}")
            
    final_score = np.mean(class_scores) # Macro average [cite: 68]
    print("-" * 65)
    print(f"MACRO COMPETITION SCORE: {final_score:.5f}")
    return final_score

In [15]:
# import copy

# epochs = 6  # Increased, but we will save the best
# best_val_loss = float('inf')
# best_model_wts = copy.deepcopy(model.state_dict())
# patience = 3  # Stop if no improvement for 3 epochs
# counter = 0
# best_comp_score = -999

# for epoch in range(epochs):
#     print(f"\nEpoch {epoch+1}/{epochs}")
    
#     # 1. Train
#     train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    
#     # 2. Validate
#     val_loss, val_preds, val_labels = validate(model, val_loader, criterion, device)

#     current_comp_score = check_improvement(val_preds, val_labels, class_names)


    
#     # 3. Checkpointing logic
#     if val_loss < best_val_loss:
#         print(f"Validation Loss Improved ({best_val_loss:.4f} ===> {val_loss:.4f}). Saving model...")
#         best_val_loss = val_loss
#         best_model_wts = copy.deepcopy(model.state_dict())
#         # torch.save(best_model_wts, 'best_model.pth')
#         counter = 0 # Reset patience
#     else:
#         counter += 1
#         print(f"No improvement in Val Loss for {counter} epoch(s).")

#     if current_comp_score > best_comp_score:
#         print(f"Score Improved ({best_comp_score:.4f} -> {current_comp_score:.4f})")
#         best_comp_score = current_comp_score
#         torch.save(model.state_dict(), 'best_model.pth')

#     print(f"Summary -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
#     # Early Stopping
#     if counter >= patience:
#         print("Early stopping triggered.")
#         break

# # Load the best weights back into the model before final evaluation/prediction
# model.load_state_dict(best_model_wts)
# print("\nBest model loaded and ready for submission!")

import copy
import torch

# --- Pre-requisites for the loop ---
# 1. Ensure you have a scheduler to refine learning as the score plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=1)

epochs = 25  # Increased slightly to allow the scheduler to work
best_comp_score = -float('inf') 
best_model_wts = copy.deepcopy(model.state_dict())
patience = 5  
counter = 0

print(f"Starting training optimized for Macro-averaged Asymmetric Cost...")

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    
    # 1. Train one epoch
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    
    # 2. Validate to get loss and predictions
    val_loss, val_preds, val_labels = validate(model, val_loader, criterion, device)

    # 3. Calculate the actual Competition Metric (The score that matters!) [cite: 61, 68]
    # This function should use the (TP - FP - 5*FN) / N formula [cite: 61]
    current_comp_score = check_improvement(val_preds, val_labels, class_names)
    
    # 4. Checkpointing logic: Prioritize the Competition Score
    if current_comp_score > best_comp_score:
        print(f"🔥 Competition Score Improved ({best_comp_score:.4f} ===> {current_comp_score:.4f}). Saving best weights...")
        best_comp_score = current_comp_score
        best_model_wts = copy.deepcopy(model.state_dict())
        
        # Save the physical file for inference later [cite: 116]
        torch.save(best_model_wts, 'best_model.pth')
        counter = 0 # Reset patience because we found a better model
    else:
        counter += 1
        print(f"No improvement in Competition Score for {counter} epoch(s).")

    # 5. Update Learning Rate based on the competition score performance
    scheduler.step(current_comp_score)
    
    print(f"Summary -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Current Score: {current_comp_score:.4f}")
    
    # 6. Early Stopping based on the competition metric
    if counter >= patience:
        print(f"Early stopping triggered. Best Competition Score reached: {best_comp_score:.4f}")
        break

# Final Step: Load the weights that gave the best competition score
model.load_state_dict(best_model_wts)
print("\n✅ Training complete. Best competition-optimized model loaded!")

Starting training optimized for Macro-averaged Asymmetric Cost...

Epoch 1/25


/tmp/ipykernel_23/3963578760.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Pathology                 | TP    | FP    | FN    | Score_c
-----------------------------------------------------------------
Atelectasis               | 61    | 222   | 409   | -4.6936
Cardiomegaly              | 21    | 120   | 99    | -4.9500
Consolidation             | 1     | 12    | 129   | -5.0462
Edema                     | 0     | 2     | 65    | -5.0308
Effusion                  | 208   | 532   | 223   | -3.3387
Emphysema                 | 0     | 1     | 34    | -5.0294
Fibrosis                  | 0     | 4     | 78    | -5.0513
Hernia                    | 0     | 0     | 7     | -5.0000
Infiltration              | 110   | 272   | 931   | -4.6273
Mass                      | 18    | 45    | 232   | -4.7480
Nodule                    | 0     | 1     | 306   | -5.0033
Pleural_Thickening        | 0     | 0     | 122   | -5.0000
Pneumonia                 | 0     | 0     | 32    | -5.0000
Pneumothorax              | 57    | 263   | 166   | -4.6457
Pneumoperitoneum          | 0     

In [16]:
# # 1. Load the best weights saved during training
# model.load_state_dict(torch.load('best_model.pth'))
# model.to(device)

# print("Final Validation Breakdown for the Best Saved Model:")
# val_loss, val_preds, val_labels = validate(model, val_loader, criterion, device)
# final_val_score = check_improvement(val_preds, val_labels, class_names)

In [17]:
# import os
# import pandas as pd
# import torch
# import numpy as np
# from torch.utils.data import Dataset, DataLoader
# from PIL import Image
# from tqdm.auto import tqdm

# # --- 1. Define Test Dataset Class ---
# class ChestXrayTestDataset(Dataset):
#     def __init__(self, csv_file, img_dir, transform=None):
#         self.df = pd.read_csv(csv_file)
#         self.img_dir = img_dir
#         self.transform = transform

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         img_id = self.df.iloc[idx, 0] # First column is 'id' [cite: 97]
#         img_path = os.path.join(self.img_dir, img_id)
        
#         # Open image and convert to RGB
#         image = Image.open(img_path).convert('RGB')
        
#         if self.transform:
#             image = self.transform(image)
            
#         return image, img_id

# # --- 2. Setup Test Loader ---
# # Use the val_transforms to ensure consistency (no augmentation for testing)
# TEST_CSV = '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/test.csv'
# test_dataset = ChestXrayTestDataset(TEST_CSV, IMAGE_DIR, transform=val_transforms)
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# # --- 3. Inference Loop ---
# # Load the best weights from your training checkpoint
# model.load_state_dict(torch.load('best_model.pth'))
# model.eval()
# model.to(device)

# test_predictions = []
# test_ids = []

# with torch.no_grad():
#     for images, ids in tqdm(test_loader, desc="Predicting Test Set"):
#         images = images.to(device)
#         outputs = model(images)
        
#         # multi-class classification: assign to one of 20 possible pathology classes [cite: 7, 15]
#         _, preds = torch.max(outputs, 1)
        
#         test_predictions.extend(preds.cpu().numpy())
#         test_ids.extend(ids)

# # --- 4. Create and Save Submission File ---
# submission_df = pd.DataFrame({'id': test_ids})

# # Assign binary indicators for each of the 20 classes [cite: 76, 83]
# for i, class_name in enumerate(class_names):
#     # Each row must contain exactly one "1" 
#     submission_df[class_name] = (np.array(test_predictions) == i).astype(int)

# # Final check: Ensure correct format [cite: 116, 117]
# submission_df.to_csv('submission.csv', index=False)
# print(f"Final submission.csv saved with {len(submission_df)} rows.")




In [18]:
import os
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import torch.nn as nn
from torchvision import transforms

# --- 1. Configuration & Paths ---
BASE_PATH = '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/'
IMAGE_DIR = os.path.join(BASE_PATH, 'images/')
TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
SAMPLE_SUB_CSV = os.path.join(BASE_PATH, 'sample_submission.csv')

# Ensure class names are in the correct order [cite: 16]
class_names = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 
    'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'Pneumoperitoneum', 
    'Pneumomediastinum', 'Subcutaneous Emphysema', 'Tortuous Aorta', 
    'Calcification of the Aorta', 'No Finding'
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. Test Dataset Class ---
class ChestXrayTestDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx, 0] # First column is 'id' [cite: 97]
        img_path = os.path.join(self.img_dir, img_id)
        
        # Open image and convert to RGB for model compatibility
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        return image, img_id

# --- 3. Setup Transforms & Loader ---
# Match the resolution used in your improved training (e.g., 320x320)
IMG_SIZE = 320 # Change to 320 if you retrained with 320
test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_dataset = ChestXrayTestDataset(TEST_CSV, IMAGE_DIR, transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# --- 4. Load Model ---
# Assuming DenseNet121 based on your previous code
import torchvision.models as models
model = models.densenet121(pretrained=False)
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 20) 

# Load the best weights from training
model.load_state_dict(torch.load('best_model.pth'))
model.to(device)
model.eval()

# --- 5. Inference with Test-Time Augmentation (TTA) ---
test_ids = []
all_probs = []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Inference (TTA)"):
        images = images.to(device)
        
        # Pass 1: Original Image
        outputs1 = model(images)
        probs1 = torch.softmax(outputs1, dim=1)
        
        # Pass 2: Horizontal Flip (TTA)
        outputs2 = model(torch.flip(images, dims=[3]))
        probs2 = torch.softmax(outputs2, dim=1)
        
        # Average probabilities to reduce variance/errors
        avg_probs = (probs1 + probs2) / 2
        
        all_probs.append(avg_probs.cpu().numpy())
        test_ids.extend(ids)

# Concatenate all batches
all_probs = np.concatenate(all_probs, axis=0)
test_predictions = np.argmax(all_probs, axis=1)

# --- 6. Create Submission File ---
submission_df = pd.DataFrame({'id': test_ids})

# Assign binary indicators (0 or 1) for each class [cite: 83]
for i, class_name in enumerate(class_names):
    # Ensure exactly one "1" per row 
    submission_df[class_name] = (test_predictions == i).astype(int)

# --- 7. Final Formatting ---
# Load sample submission to ensure exact column alignment 
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
submission_df = submission_df[sample_sub.columns]

submission_df.to_csv('submission.csv', index=False)
print(f"Final submission.csv saved with {len(submission_df)} rows.")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Inference (TTA):   0%|          | 0/532 [00:00<?, ?it/s]

Final submission.csv saved with 17015 rows.


In [19]:
# import os
# import copy
# import numpy as np
# import pandas as pd
# from PIL import Image
# from tqdm.auto import tqdm

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
# from torchvision import transforms, models

# from sklearn.model_selection import train_test_split

In [20]:
# BASE_PATH = '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/'
# IMAGE_DIR = os.path.join(BASE_PATH, 'images/')
# TRAIN_CSV = os.path.join(BASE_PATH, 'train.csv')
# TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
# SAMPLE_SUB = os.path.join(BASE_PATH, 'sample_submission.csv')

# IMG_SIZE = 224
# BATCH_SIZE = 32
# EPOCHS = 20
# PATIENCE = 5

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [21]:
# class_names = [
#     'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 
#     'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 
#     'Pleural_Thickening', 'Pneumonia', 'Pneumothorax', 'Pneumoperitoneum', 
#     'Pneumomediastinum', 'Subcutaneous Emphysema', 'Tortuous Aorta', 
#     'Calcification of the Aorta', 'No Finding'
# ]

In [22]:
# train_df = pd.read_csv(TRAIN_CSV)

# train_df['label_idx'] = np.argmax(train_df[class_names].values, axis=1)

# train_data, val_data = train_test_split(
#     train_df,
#     test_size=0.2,
#     stratify=train_df['label_idx'],
#     random_state=42
# )

In [23]:
# class ChestXrayDataset(Dataset):
#     def __init__(self, df, img_dir, transform=None):
#         self.df = df
#         self.img_dir = img_dir
#         self.transform = transform
#         self.labels = df[class_names].values

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         img_name = self.df.iloc[idx]['id']
#         img_path = os.path.join(self.img_dir, img_name)

#         image = Image.open(img_path).convert('RGB')
#         label = np.argmax(self.labels[idx])

#         if self.transform:
#             image = self.transform(image)

#         return image, label

In [24]:
# train_transforms = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(10),
#     transforms.RandomAffine(degrees=0, translate=(0.05,0.05)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
# ])

# val_transforms = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
# ])

In [25]:
# class_counts = train_data['label_idx'].value_counts().sort_index().values
# class_weights_sampler = 1. / class_counts

# sample_weights = train_data['label_idx'].map(lambda x: class_weights_sampler[x]).values
# sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

In [26]:
# train_dataset = ChestXrayDataset(train_data, IMAGE_DIR, train_transforms)
# val_dataset = ChestXrayDataset(val_data, IMAGE_DIR, val_transforms)

# train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
# val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [27]:
# model = models.densenet121(pretrained=True)

# # Freeze backbone
# for param in model.features.parameters():
#     param.requires_grad = False

# num_ftrs = model.classifier.in_features
# model.classifier = nn.Linear(num_ftrs, 20)

# model = model.to(device)

In [28]:
# class_counts_array = train_df[class_names].sum().values
# total_samples = len(train_df)

# weights = np.sqrt(total_samples / (class_counts_array + 1e-6))

# # BOOST disease classes
# weights[:-1] = weights[:-1] * 6.0

# weights = weights / np.mean(weights)

# class_weights = torch.FloatTensor(weights).to(device)

In [29]:
# class FocalLoss(nn.Module):
#     def __init__(self, alpha=None, gamma=2):
#         super().__init__()
#         self.alpha = alpha
#         self.gamma = gamma

#     def forward(self, inputs, targets):
#         ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
#         pt = torch.exp(-ce_loss)
#         loss = ((1 - pt) ** self.gamma) * ce_loss
#         return loss.mean()

# criterion = FocalLoss(alpha=class_weights, gamma=2)

In [30]:
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#     optimizer, mode='max', patience=2, factor=0.2
# )

In [31]:
# def train_one_epoch(model, loader):
#     model.train()
#     total_loss = 0

#     for images, labels in loader:
#         images, labels = images.to(device), labels.to(device)

#         optimizer.zero_grad()
#         outputs = model(images)
#         loss = criterion(outputs, labels)

#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     return total_loss / len(loader)

In [32]:
# def validate(model, loader):
#     model.eval()
#     all_preds, all_labels = [], []

#     with torch.no_grad():
#         for images, labels in loader:
#             images = images.to(device)
#             outputs = model(images)

#             preds = torch.argmax(outputs, dim=1)

#             all_preds.extend(preds.cpu().numpy())
#             all_labels.extend(labels.numpy())

#     return np.array(all_preds), np.array(all_labels)

In [33]:
# def competition_score(preds, labels):
#     scores = []

#     for i in range(len(class_names)):
#         tp = np.sum((preds==i) & (labels==i))
#         fp = np.sum((preds==i) & (labels!=i))
#         fn = np.sum((preds!=i) & (labels==i))
#         nc = np.sum(labels==i)

#         if nc > 0:
#             score = (tp - fp - 5*fn) / nc
#             scores.append(score)

#     return np.mean(scores)

In [34]:
# best_score = -999
# best_wts = copy.deepcopy(model.state_dict())

# counter = 0

# for epoch in range(EPOCHS):
#     print(f"\nEpoch {epoch+1}")

#     train_loss = train_one_epoch(model, train_loader)
#     preds, labels = validate(model, val_loader)

#     score = competition_score(preds, labels)

#     print(f"Loss: {train_loss:.4f} | Score: {score:.4f}")

#     if score > best_score:
#         best_score = score
#         best_wts = copy.deepcopy(model.state_dict())
#         torch.save(best_wts, 'best_model.pth')
#         counter = 0
#         print("🔥 Improved!")
#     else:
#         counter += 1

#     scheduler.step(score)

#     # UNFREEZE after 3 epochs
#     if epoch == 3:
#         for param in model.features.parameters():
#             param.requires_grad = True

#     if counter >= PATIENCE:
#         print("Early stopping")
#         break

# model.load_state_dict(best_wts)

In [35]:
# class TestDataset(Dataset):
#     def __init__(self, csv_file, img_dir, transform):
#         self.df = pd.read_csv(csv_file)
#         self.img_dir = img_dir
#         self.transform = transform

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         img_id = self.df.iloc[idx,0]
#         img_path = os.path.join(self.img_dir, img_id)

#         image = Image.open(img_path).convert('RGB')

#         if self.transform:
#             image = self.transform(image)

#         return image, img_id

In [36]:
# test_dataset = TestDataset(TEST_CSV, IMAGE_DIR, val_transforms)
# test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# model.eval()

# all_probs = []
# test_ids = []

# with torch.no_grad():
#     for images, ids in tqdm(test_loader):
#         images = images.to(device)

#         # original
#         p1 = torch.softmax(model(images), dim=1)

#         # flip
#         p2 = torch.softmax(model(torch.flip(images, [3])), dim=1)

#         # rotate
#         p3 = torch.softmax(model(torch.rot90(images, 1, [2,3])), dim=1)

#         probs = (p1 + p2 + p3) / 3

#         all_probs.append(probs.cpu().numpy())
#         test_ids.extend(ids)

# all_probs = np.concatenate(all_probs)
# preds = np.argmax(all_probs, axis=1)

In [37]:
# submission = pd.DataFrame({'id': test_ids})

# for i, c in enumerate(class_names):
#     submission[c] = (preds == i).astype(int)

# sample = pd.read_csv(SAMPLE_SUB)
# submission = submission[sample.columns]

# submission.to_csv('submission.csv', index=False)

# print("✅ Submission ready!")